In [1]:
import pandas as pd
import os
import gc
import numpy as np
from readlif.reader import LifFile
import tifffile

In [ ]:

def lif_to_tiff_pipeline(lif_path: str, output_dir: str, target_channel: int = 0):
    """
    将 .lif 容器中的所有时空张量子集裂解为 ImageJ 兼容的 OME-TIFF 视频。
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # 1. 实例化 LIF 解析器 (此时不发生张量数据的 RAM 载入)
    lif_obj = LifFile(lif_path)
    print(f"[I/O] 成功挂载 LIF 容器，共检测到 {lif_obj.num_images} 个 FOV 节点。")

    # 2. 遍历物理流形树
    for img_idx, img in enumerate(lif_obj.get_iter_image()):

        # 【核心修正】提取 dims 对象并解析具体属性标量
        dims = img.info['dims']
        T = dims.t
        Y = dims.y
        X = dims.x
        
        # 【核心修正】解构 bit_depth 元组
        bit_depth = img.info['bit_depth'][0] if isinstance(img.info['bit_depth'], tuple) else img.info['bit_depth']
        
        # 仅处理包含时间维度的动态视频序列
        if T <= 1:
            print(f"[-] 跳过单帧静态图像: {img.name}")
            continue
            
        # 物理标度先验提取 (用于 AQuA2 的 spatialRes)
        scale_x = img.info['scale_n'][1] if img.info['scale_n'][1] else 1.0
        
        # 根据物理位深映射 numpy 底层数据类型
        np_dtype = np.uint16 if bit_depth > 8 else np.uint8
        
        # 3. 在连续内存中预分配该 FOV 的零矩阵 (T, Y, X)
        print(f"[Compute] 正在反序列化 FOV: {img.name} | 维度: {T}x{Y}x{X} | {bit_depth}-bit")
        movie_tensor = np.zeros((T, Y, X), dtype=np_dtype)
        
        # 4. 沿时间轴正交投影，提取目标通道切片
        for t in range(T):
            # 获取单帧的底层 PIL 对象并强制转化为 numpy 数组
            frame = img.get_frame(z=0, t=t, c=target_channel)
            movie_tensor[t, :, :] = np.array(frame)
            
        # 5. 构造输出绝对路径 (处理非规范字符)
        safe_name = img.name.replace("/", "_").replace(" ", "_")
        out_path = os.path.join(output_dir, f"{safe_name}.tif")
        
        # 6. 利用 tifffile 执行落盘，强制注入 ImageJ 元数据以保证 MATLAB 读取不发生维度坍缩
        tifffile.imwrite(
            out_path,
            movie_tensor,
            imagej=True,
            resolution=(1./scale_x, 1./scale_x),
            metadata={'axes': 'TYX', 'unit': 'um'}
        )
        
        # 7. 显式释放内存指针，规避连续提取导致的内存泄漏
        del movie_tensor
        gc.collect()

if __name__ == "__main__":
    LIF_FILE = "3.17.26 organoid day4 coculture +- ABO.lif"   # 替换为您的输入路径
    OUTPUT_FOLDER = "coculture_3_17/"     # 替换为您的输出路径
    
    # 假定星形胶质细胞钙瞬变数据位于 Channel 0
    lif_to_tiff_pipeline(LIF_FILE, OUTPUT_FOLDER, target_channel=0)

[I/O] 成功挂载 LIF 容器，共检测到 6 个 FOV 节点。
[Compute] 正在反序列化 FOV: Coculture rep1 +ABO | 真实维度: 441x512x512 | 8-bit
[Compute] 正在反序列化 FOV: Coculture rep1 control | 真实维度: 441x512x512 | 8-bit
[Compute] 正在反序列化 FOV: Coculture rep2 +ABO | 真实维度: 441x512x512 | 8-bit
[Compute] 正在反序列化 FOV: Coculture rep2+control | 真实维度: 441x512x512 | 8-bit
[Compute] 正在反序列化 FOV: Coculture rep3+control | 真实维度: 441x512x512 | 8-bit
[Compute] 正在反序列化 FOV: Coculture rep3 +ABO | 真实维度: 441x512x512 | 8-bit


In [14]:
import pandas as pd
params = pd.read_csv("/home/crnlqz/krenciklab/AQuA2-main/cfg/parameters_for_batch.csv",header=0)
params

,Unnamed: 0,Name,Variable,Type,File1,File2,File3,File4,File5,File6,File7,File8,File9,File10,File11,File12
0,0,Registration mode,registrateCorrect,preprocessing,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
1,1,Bleach correction mode,bleachCorrect,preprocessing,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
2,2,Median filter radius (For salt and pepper noise),medSmo,preprocessing,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
3,3,Spatial smoothing level,smoXY,preprocessing,0.50,0.50,0.50,0.50,0.50,0.50,0.50,0.50,0.50,0.50,0.50,0.50
4,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,5,Active voxels threshold scale,thrARScl,activeregion,2.00,2.00,2.00,2.00,2.00,2.00,3.00,3.00,3.00,3.00,3.00,3.00
6,6,Minimum duration of signal,minDur,activeregion,3.00,3.00,3.00,3.00,3.00,3.00,5.00,5.00,5.00,5.00,5.00,5.00
7,7,Minimum size,minSize,activeregion,10.00,10.00,10.00,10.00,10.00,10.00,20.00,20.00,20.00,20.00,20.00,20.00
8,8,Maximum size,maxSize,activeregion,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf
9,9,Circularity threshold,circularityThr,activeregion,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


In [11]:
# 2. 定义您需要局部覆写的松弛参数拓扑
loose_params = {
    "smoXY": 0.5, 
    "thrARScl": 2, 
    "zThre": 1.5,
    "minDur": 3,
    "minSize": 10,
    "sigThr":10,
    "detectGlo":0,
    "SpatialRes":0.92,
    "frameRate":1.36,

}

# 3. 遍历目标子集，执行安全的局部寻址与标量更新
for param_name, new_value in loose_params.items():
    # 构建布尔掩码：仅定位 Variable 列精确等于 param_name 的行
    mask = (params['Variable'] == param_name)
    
    # 仅针对掩码范围内的 File1 列进行标量赋值，其余行严格保持底层不变
    params.loc[mask, ['File1','File2','File3','File4','File5','File6']] = new_value

params

,Name,Variable,Type,File1,File2,File3,File4,File5,File6,File7,File8,File9,File10,File11,File12
0,Registration mode,registrateCorrect,preprocessing,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
1,Bleach correction mode,bleachCorrect,preprocessing,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
2,Median filter radius (For salt and pepper noise),medSmo,preprocessing,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
3,Spatial smoothing level,smoXY,preprocessing,0.50,0.50,0.50,0.50,0.50,0.50,0.50,0.50,0.50,0.50,0.50,0.50
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Active voxels threshold scale,thrARScl,activeregion,2.00,2.00,2.00,2.00,2.00,2.00,3.00,3.00,3.00,3.00,3.00,3.00
6,Minimum duration of signal,minDur,activeregion,3.00,3.00,3.00,3.00,3.00,3.00,5.00,5.00,5.00,5.00,5.00,5.00
7,Minimum size,minSize,activeregion,10.00,10.00,10.00,10.00,10.00,10.00,20.00,20.00,20.00,20.00,20.00,20.00
8,Maximum size,maxSize,activeregion,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf
9,Circularity threshold,circularityThr,activeregion,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


In [15]:
params = params.iloc[:,1:]
params

,Name,Variable,Type,File1,File2,File3,File4,File5,File6,File7,File8,File9,File10,File11,File12
0,Registration mode,registrateCorrect,preprocessing,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
1,Bleach correction mode,bleachCorrect,preprocessing,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
2,Median filter radius (For salt and pepper noise),medSmo,preprocessing,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
3,Spatial smoothing level,smoXY,preprocessing,0.50,0.50,0.50,0.50,0.50,0.50,0.50,0.50,0.50,0.50,0.50,0.50
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Active voxels threshold scale,thrARScl,activeregion,2.00,2.00,2.00,2.00,2.00,2.00,3.00,3.00,3.00,3.00,3.00,3.00
6,Minimum duration of signal,minDur,activeregion,3.00,3.00,3.00,3.00,3.00,3.00,5.00,5.00,5.00,5.00,5.00,5.00
7,Minimum size,minSize,activeregion,10.00,10.00,10.00,10.00,10.00,10.00,20.00,20.00,20.00,20.00,20.00,20.00
8,Maximum size,maxSize,activeregion,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf
9,Circularity threshold,circularityThr,activeregion,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


In [16]:
params.to_csv("/home/crnlqz/krenciklab/AQuA2-main/cfg/parameters_for_batch.csv",index=False)

analysis output of aqua2

In [9]:
out_dir = '/home/crnlqz/krenciklab/coculture_3_17_out'

In [10]:
out_dir_list = [os.path.join(out_dir,dir) for dir in os.listdir(out_dir)]
out_dir_list

['/home/crnlqz/krenciklab/coculture_3_17_out/Coculture_rep1_+ABO_results',
 '/home/crnlqz/krenciklab/coculture_3_17_out/Coculture_rep1_control_results',
 '/home/crnlqz/krenciklab/coculture_3_17_out/Coculture_rep2+control_results',
 '/home/crnlqz/krenciklab/coculture_3_17_out/Coculture_rep2_+ABO_results',
 '/home/crnlqz/krenciklab/coculture_3_17_out/Coculture_rep3+control_results',
 '/home/crnlqz/krenciklab/coculture_3_17_out/Coculture_rep3_+ABO_results']

In [17]:
os.listdir(out_dir_list[1])[1]

'Coculture_rep1_control_AQuA2_Ch1.csv'

In [20]:
'''['Coculture_rep1_control_AQuA2.mat',
 'Coculture_rep1_control_AQuA2_Ch1.csv',
 'Coculture_rep1_control_AQuA2_Ch1_curves.xlsx',
 'Coculture_rep1_control_AQuA2_Movie.tif',
 'Coculture_rep1_control_AQuA2_risingMaps_Ch1']'''

path = os.path.join(out_dir_list[0],os.listdir(out_dir_list[0])[2])

In [ ]:
csv = pd.ExcelFile(path)
